# Finding the Ground State Energy of a Molecule Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Finding the Ground State Energy of a Molecule" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits
from test_GroundStateEnergy import HamiltonianTerm

## Problem 1. Single-qubit Pauli product rotation

Remember that the rotation gates $R_x$, $R_y$, and $R_z$ are expressed as exponents of Pauli matrices:

$$R_x(\theta) = \exp(-i \tfrac{\theta}{2} X), R_y(\theta) = \exp(-i \tfrac{\theta}{2} Y), R_z(\theta) = \exp(-i \tfrac{\theta}{2} Z)$$

One possible solution is to decide which of the rotation gates to use based on the values of the parameters `is_x` and `is_z`, and apply that gate with the rotation angle $-2\theta$ (in radians).

In [ ]:
def single_qubit_ppr_1(reg: Qubits, theta: float, is_x: bool, is_z: bool) -> None:
    from psiqdk.workbench import units
    angle = -2 * theta * units.rad
    if is_x and not is_z:
        reg.rx(angle)
    elif is_x and is_z:
        reg.ry(angle)
    else:
        reg.rz(angle)

Alternatively, you can use the built-in method `ppr` of the `Qubits` class that implements exactly the Pauli product rotation. The arguments `x_mask` and `z_mask` which specify the Pauli matrices to use for each qubit are the same as `is_x` and `is_z` for our single-qubit scenario.

In [ ]:
def single_qubit_ppr_2(reg: Qubits, theta: float, is_x: bool, is_z: bool) -> None:
    from psiqdk.workbench import units
    reg.ppr(-theta * units.rad, int(is_x), int(is_z))

## Problem 2. Two-qubit Pauli product rotation

Similarly to the previous problem, the built-in method `ppr` of the `Qubits` class implements the Pauli product rotation. Conveniently, it takes the mask arguments in the same format as they are given in this task!

In [ ]:
def two_qubit_ppr(reg: Qubits, theta: float, x_mask: bool, z_mask: bool) -> None:
    from psiqdk.workbench import units
    reg.ppr(-theta * units.rad, x_mask, z_mask)

## Problem 3. Implement one Trotter step (first-order Trotter decomposition)

We already know how to apply each term of the decomposition: it's just the previous problem with the masks given in the term description and `theta` argument equal to $-t c_j$. Following the instructions from the problem description, we need to iterate through the terms and apply each of them in the order in which they are given.

In [ ]:
def trotter_step(reg: Qubits, t: float, terms: list[HamiltonianTerm]) -> None:
    for term in terms:
        two_qubit_ppr(reg, -t * term.c, term.x_mask, term.z_mask)

## Problem 4. Implement first-order Trotter decomposition

Following the definition of first-order Trotter decomposition, you need to call the Trotter step you implemented in the previous problem $n$ times, each one using evolution time $\frac{t}{n}$.

In [ ]:
def trotter_approximation(reg: Qubits, t: float, terms: list[HamiltonianTerm], n: int) -> None:
    for _ in range(n):
        trotter_step(reg, t / n, terms)